# IterCOMP on Colab ? original architecture
Run cells manually in order. These notebooks have never executed model inference. Enable a GPU, accept the Mistral/Llama checkpoint terms where required, and configure paths below. No training is performed. Default model pair: Mistral-7B controller, Llama-3-8B-Instruct reader, NF4/FP16. Compression and reading run in separate child processes so model weights are released between stages.

The exact controller checkpoint, instruction-tuned reader variant, and generation settings are reproduction assumptions. See `docs/reproduction_notes.md`. Start with the existing smoke split; evaluation is a separate deliberate run.

## 1. Runtime and persistent path configuration
Edit the placeholder paths. The prepared dataset splits must already exist; this notebook never regenerates them.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil, gc
# Configure Transformers before optional-framework imports; child processes inherit this.
os.environ.update(USE_TORCH='1', USE_TF='0', USE_FLAX='0')
MOUNT_DRIVE = True  # Optional: persistent outputs on your Drive
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
# Upload/extract your repository at REPO, or change it to an existing checkout.
REPO = Path('/content/musique_itercomp')
DATA_DIR = REPO / 'data' / 'benchmark'  # configurable prepared inputs/gold/manifest
OUTPUT_ROOT = Path('/content/drive/MyDrive/itercomp_runs') if MOUNT_DRIVE else Path('/content/itercomp_runs')
INTERNET = True
# Keep transient model caches separate from persistent experiment outputs.
os.environ.setdefault('HF_HOME', '/content/hf_cache')


## 2. GPU availability and memory
Dual T4 devices are separate memory pools. CUDA:0 is the default; choose CUDA:1 explicitly for the encoder if available.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU runtime; CPU fallback is disabled')
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(i, torch.cuda.get_device_name(i), 'free GiB', round(free/2**30, 2), 'total GiB', round(total/2**30, 2))
print('Existing torch:', torch.__version__, 'CUDA:', torch.version.cuda)


## 3. Repository setup

In [ ]:
if not (REPO / 'pyproject.toml').is_file():
    raise FileNotFoundError('Upload/extract or clone the implemented repository at REPO before continuing')
os.chdir(REPO)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## 4. Install inference dependencies
The PyTorch-only pipeline disables TensorFlow/Flax imports. Requirements preserve platform Protobuf and use bitsandbytes 0.49.2 for CUDA 13. If an older installation downgraded Protobuf, start a fresh runtime or restore its original version first. Restart the kernel after changing already-imported dependencies, then rerun from runtime configuration.
Intended runtime: Linux, Python 3.10?3.12, existing CUDA PyTorch 2.6/2.7. The requirements file does not reinstall torch. GPU compatibility has not been validated in this project. Restart the notebook runtime if imports were already loaded before package changes. Offline Kaggle sessions need dependencies preinstalled or an attached wheel directory.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-gpu.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], check=True)
# Validate the installed CUDA extension in a fresh interpreter, before model downloads.
# This check imports libraries only; it never downloads or loads pretrained weights.
subprocess.run([sys.executable, '-c', "import torch, bitsandbytes as bnb; from bitsandbytes.cextension import lib; print('torch CUDA:', torch.version.cuda, 'bitsandbytes:', bnb.__version__); assert lib is not None and getattr(lib, 'compiled_with_cuda', False) is True, 'Native bitsandbytes CUDA backend missing. Reinstall bitsandbytes==0.49.2 with --force-reinstall --no-cache-dir --no-deps, restart the kernel, and check the working requirements file.'"], check=True)

# The running notebook kernel may not process a newly installed editable .pth
# until restart. Make the src-layout package available in this kernel now.
import importlib
source_dir = (REPO / 'src').resolve()
if not (source_dir / 'itercomp' / '__init__.py').is_file():
    raise FileNotFoundError(f'Expected the IterCOMP package under {source_dir}; check REPO')
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))
importlib.invalidate_caches()

from itercomp.cloud import runtime_check
print(json.dumps(runtime_check(), indent=2))


## 5. Dataset paths and identity
Inspect schema only here. Gold is opened later exclusively by CPU evaluation or the explicit Oracle baseline.

In [ ]:
SPLIT = 'smoke'  # smoke | calibration | evaluation
INPUTS = DATA_DIR / SPLIT / 'inputs.jsonl'
GOLD = DATA_DIR / SPLIT / 'gold.jsonl'
if not INPUTS.is_file() or not GOLD.is_file():
    raise FileNotFoundError('Configure DATA_DIR for the existing prepared benchmark directory')
from itercomp.serialization import file_hash
from itercomp.dataset import load_inputs
print('Input SHA256:', file_hash(INPUTS), 'samples:', len(load_inputs(INPUTS)))


## 6. Hugging Face authentication
Create a read token at https://huggingface.co/settings/tokens. Add it to Colab Secrets as HF_TOKEN and enable this notebook's access. Existing environment/cached logins are reused. Request Llama checkpoint access with the same Hugging Face account; a valid token alone does not grant gated access. Never print tokens.

In [ ]:
# Authentication is independent of the repository version or cached modules.
from huggingface_hub import HfApi, get_token
token = get_token()
if not token:
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception:
        print('Add HF_TOKEN to Colab Secrets and enable notebook access. Public/local checkpoints may work without it.')
if token:
    os.environ['HF_TOKEN'] = token.strip()
    os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '0'  # reader subprocess inherits credentials
    if INTERNET:
        HfApi(token=os.environ['HF_TOKEN']).whoami()
        print('Hugging Face authentication verified. Gated-model approval is separate.')
    else:
        print('HF_TOKEN found; online verification skipped.')


## 7. Experiment configuration
Names are reader then controller. Use the same Mistral compression output for Llama/Mistral and Qwen/Mistral reader comparisons. Reader changes leave the saved compressed context unchanged. Pin `REVISIONS` to commit hashes for repeatable downloads. For offline Kaggle, set each checkpoint to its directory under `/kaggle/input`; include BGE sparse heads.

In [ ]:
PROFILE = 'itercomp_llama_mistral'
METHOD = 'itercomp'  # raw | oracle | itercomp
# Example offline paths: '/kaggle/input/bge-m3/pytorch/default/1', etc.
CHECKPOINTS = {'embedding': None, 'controller': None, 'reader': None}
REVISIONS = {'embedding': None, 'controller': None, 'reader': None}
ENCODER_DEVICE = 'cuda:0'  # 'cuda:1' if explicitly using a second GPU
CONTROLLER_DEVICE = 'cuda:0'
READER_DEVICE = 'cuda:0'
LIMIT = 2  # manual real-model smoke validation first; None processes the selected split
# Separate names for every dataset/controller/config/subset and reader experiment.
CONTROLLER_LABEL = PROFILE.rsplit('_', 1)[-1]
RUN_TAG = f'limit{LIMIT}' if LIMIT is not None else 'full'
COMPRESSION_DIR = OUTPUT_ROOT / f'{SPLIT}_{METHOD}_{CONTROLLER_LABEL}_compression_{RUN_TAG}'
READER_DIR = OUTPUT_ROOT / f'{SPLIT}_{METHOD}_{PROFILE}_{RUN_TAG}'
RESOLVED_CONFIG = OUTPUT_ROOT / f'{PROFILE}_notebook.yaml'
from itercomp.cloud import notebook_config
cfg = notebook_config(REPO / 'configs' / f'{PROFILE}.yaml', RESOLVED_CONFIG,
                      CHECKPOINTS, REVISIONS, INTERNET, ENCODER_DEVICE, CONTROLLER_DEVICE, READER_DEVICE)


## 8. Verify model access without loading weights
Online mode checks model metadata and authenticated HEAD access to config.json before inference. No checkpoint weights are downloaded here. Offline mode checks local configuration/head files. Config access is an early gate check; weight loading may still encounter other errors.

In [ ]:
from itercomp.cloud import verify_model_access
roles = ('embedding', 'controller', 'reader') if METHOD == 'itercomp' else ('reader',)
print(json.dumps(verify_model_access(cfg, roles=roles), indent=2))
from huggingface_hub import get_hf_file_metadata, hf_hub_url, get_token
for role in roles:
    model = getattr(cfg, role)
    if model.backend in ('mock', 'openai_compatible') or Path(model.model_id).is_dir() or model.local_files_only:
        continue
    get_hf_file_metadata(hf_hub_url(model.model_id, 'config.json', revision=model.revision), token=get_token())
    print(role, 'checkpoint configuration access verified')


## 9. Compression execution ? manual GPU action
This cell intentionally loads only BGE-M3 and the controller for IterCOMP. Raw/Oracle load neither. Initial pass counts as iteration 1. Set LIMIT and choose a new output directory for a different subset.

In [ ]:
base = [sys.executable, 'scripts/run_experiment.py', '--config', str(RESOLVED_CONFIG),
        '--inputs', str(INPUTS), '--split', SPLIT]
compression_cmd = base + ['--stage', 'compress', '--method', METHOD, '--output-dir', str(COMPRESSION_DIR)]
if METHOD == 'oracle':
    compression_cmd += ['--oracle-gold', str(GOLD)]
if LIMIT is not None:
    compression_cmd += ['--limit', str(LIMIT)]
# Add --resume only when continuing the same run and subset.
subprocess.run(compression_cmd, check=True)


## 10. Save and inspect compression artifacts
Every completed sample is flushed/fsynced. Full rankings are enabled and can be large. Preserve this directory before the free runtime ends.

In [ ]:
COMPRESSION_RESULTS = COMPRESSION_DIR / 'compression_results.jsonl'
metadata = json.loads((COMPRESSION_DIR / 'run_metadata.json').read_text())
print({k: metadata.get(k) for k in ('status', 'complete_split', 'completed_count', 'output_sha256')})


## 11. Release compression models
The compression subprocess has exited, so its model references and CUDA context are gone. The runner also explicitly closes models, collects garbage, empties unused CUDA allocator memory, and records memory diagnostics before exit.

In [ ]:
from itercomp.telemetry import release_memory
print(json.dumps(release_memory(), indent=2))


## 12. Reader execution ? manual GPU action
Loads only the reader. Validates artifact data/config/prompt/source identities before loading weights. Counts raw and compressed tokens using the chosen reader tokenizer. A different reader profile can reuse these artifacts when all compression settings match.

In [ ]:
reader_cmd = base + ['--stage', 'read', '--compression-results', str(COMPRESSION_RESULTS),
                     '--output-dir', str(READER_DIR)]
subprocess.run(reader_cmd, check=True)
PREDICTIONS = READER_DIR / 'predictions.jsonl'


## 13. CPU-only evaluation
For LIMIT/subset runs this explicitly creates a matching gold subset and marks the reports partial. This cell performs no model inference.

In [ ]:
REPORT_DIR = READER_DIR / 'reports'
evaluation_cmd = [sys.executable, 'scripts/evaluate_run.py', '--gold', str(GOLD),
                  '--predictions', str(PREDICTIONS), '--out', str(REPORT_DIR)]
if LIMIT is not None:
    evaluation_cmd += ['--allow-partial']
subprocess.run(evaluation_cmd, check=True)


## 14. Export artifacts
Kaggle: save a notebook version with outputs or download/export the archive. Colab: use Drive outputs or explicitly download; `/content` does not persist across runtime termination. Exclude checkpoint caches from results.

In [ ]:
archive = shutil.make_archive(str(OUTPUT_ROOT.parent / (OUTPUT_ROOT.name + '_export')), 'zip', OUTPUT_ROOT)
print('Export:', archive)
# Optional Colab download:
# from google.colab import files
# files.download(archive)


## 15. Resume after interruption
Recreate identical path/config variables, authenticate, and verify access. Resume compression first if its metadata says interrupted. Completed IDs are skipped. Final incomplete JSONL bytes are quarantined and repaired; duplicate IDs or interior corruption are rejected. Reader and compression stages resume independently. Changing limits, subsets, prompts or relevant model settings requires a new output directory.

In [ ]:
# Run ONLY the stage you need to resume:
# subprocess.run(compression_cmd + ['--resume'], check=True)
# subprocess.run(reader_cmd + ['--resume'], check=True)
# Use a fresh report directory if scoring again; existing reports are preserved.
